# Parameter Golf — Experiments

Run `parameter_golf_setup.ipynb` first to clone repos, install deps, and download data.

This notebook handles:
1. Quick Pull (sync latest code)
2. GPU detection + config selection
3. Baseline runs
4. Individual loss sweeps
5. Winner confirmation (3 seeds)
6. Stacking experiments
7. Report generation
8. **Error analysis** — understand WHERE the model fails

## 0. Quick Pull + Verify

Always run this first to get the latest code from GitHub.

In [ ]:
import shutil, os, glob, torch

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)

OFFICIAL = '/content/parameter-golf'
AUX = '/content/parameter-golf-aux'

# Clone repos if not present (in case setup notebook wasn't run)
%cd /content
if not os.path.exists(OFFICIAL):
    !git clone --depth 1 https://github.com/openai/parameter-golf.git
if not os.path.exists(AUX):
    !git clone https://github.com/jamesconde/parameter-golf-aux.git

# Install deps if needed
!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard 2>/dev/null

# Download data if not present
if not os.path.exists(f'{OFFICIAL}/data/datasets/fineweb10B_sp1024/fineweb_val_000000.bin'):
    %cd {OFFICIAL}
    !python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1

# Pull latest from our repo
%cd {AUX}
!git pull

# Copy all our files into the official repo
shutil.copy2(f'{AUX}/train_gpt_aux.py', f'{OFFICIAL}/train_gpt_aux.py')
shutil.copy2(f'{AUX}/train_gpt_sota.py', f'{OFFICIAL}/train_gpt_sota.py')
if os.path.exists(f'{OFFICIAL}/aux_losses'):
    shutil.rmtree(f'{OFFICIAL}/aux_losses')
shutil.copytree(f'{AUX}/aux_losses', f'{OFFICIAL}/aux_losses')
# Copy ALL scripts
for src_file in glob.glob(f'{AUX}/scripts/*.py'):
    os.makedirs(f'{OFFICIAL}/scripts', exist_ok=True)
    shutil.copy2(src_file, f'{OFFICIAL}/scripts/{os.path.basename(src_file)}')
# Copy ALL sweep configs
for cfg_file in glob.glob(f'{AUX}/experiments/sweep_config*.json'):
    os.makedirs(f'{OFFICIAL}/experiments', exist_ok=True)
    shutil.copy2(cfg_file, f'{OFFICIAL}/experiments/{os.path.basename(cfg_file)}')

# Ensure logs symlink to Drive
logs_dir = f'{OFFICIAL}/logs'
if os.path.exists(logs_dir) and not os.path.islink(logs_dir):
    for f_name in os.listdir(logs_dir):
        shutil.copy2(f'{logs_dir}/{f_name}', f'{DRIVE_DIR}/logs/{f_name}')
    shutil.rmtree(logs_dir)
if not os.path.exists(logs_dir):
    os.symlink(f'{DRIVE_DIR}/logs', logs_dir)

# Clear stale logs from previous buggy runs
stale_count = 0
for logfile in glob.glob(f'{DRIVE_DIR}/logs/*.txt'):
    with open(logfile) as lf:
        content = lf.read()
    if 'step:1/' not in content and 'warmup_step:1/' not in content:
        os.remove(logfile)
        stale_count += 1
if stale_count > 0:
    print(f'Cleared {stale_count} stale logs from crashed runs')

# Verify
%cd {OFFICIAL}
!python3 -c "import py_compile; py_compile.compile('train_gpt_aux.py', doraise=True); py_compile.compile('train_gpt_sota.py', doraise=True); print('Scripts: OK')"
!python3 -c "from aux_losses import focal_cross_entropy, inter_layer_decorrelation_loss, representation_rank_loss, unigram_kl_loss; print('Aux losses: OK')"

# GPU info + config selection
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\nGPU: {gpu_name} ({vram_gb:.0f}GB)')

if vram_gb >= 70:
    CONFIG = 'experiments/sweep_config_a100_80g.json'
    print(f'Config: A100-80GB (786K batch tokens, 500 iters)')
elif vram_gb >= 35:
    CONFIG = 'experiments/sweep_config_a100_40g.json'
    print(f'Config: A100-40GB (393K batch tokens, 500 iters)')
elif vram_gb >= 14:
    CONFIG = 'experiments/sweep_config_t4.json'
    print(f'Config: T4 (131K batch tokens, 300 iters)')
else:
    CONFIG = 'experiments/sweep_config_t4.json'
    print(f'Config: T4/small GPU (131K batch tokens, 300 iters)')

print('\nReady.')

## 1. Baselines (run first)

Runs both baselines × 3 seeds:
- `baseline_sota`: unmodified SOTA script (overhead check)
- `baseline`: our script with aux losses disabled (reference for all comparisons)

In [ ]:
%cd /content/parameter-golf
!python3 scripts/experiment_runner.py --config {CONFIG} --filter "baseline" --seeds 42,1337,7

## 2. Screen Individual Losses (1 seed each)

Quick screen: 14 experiments × 1 seed. Identifies which losses and lambdas are promising.

In [ ]:
%cd /content/parameter-golf
!python3 scripts/experiment_runner.py --config {CONFIG} --filter "focal|decorr|rank|unigram" --seeds 42

## 3. Check Intermediate Results

Generate report from all completed runs so far.

In [ ]:
%cd /content/parameter-golf
!python3 scripts/experiment_runner.py --report-only

from IPython.display import Markdown, display
if os.path.exists('experiments/results_auto.md'):
    display(Markdown(open('experiments/results_auto.md').read()))

## 4. Confirm Winners (3 seeds)

Edit the filter below to match the best experiments from Step 2.

In [ ]:
# EDIT THIS: replace with the winning experiment names from Step 2
WINNERS = "focal_g1|decorr_001"

%cd /content/parameter-golf
!python3 scripts/experiment_runner.py --config {CONFIG} --filter "{WINNERS}" --seeds 42,1337,7

## 5. Stack Winners

Combine the aux losses that individually improved BPB.

In [ ]:
# EDIT THIS: fill in the best lambda values from Steps 2-4
%cd /content/parameter-golf

import json, os

# Load base config and add stacking experiments
with open(CONFIG) as f:
    cfg = json.load(f)

# Define stacking experiments — edit these based on your results
stack_experiments = [
    {
        "name": "stack_focal_decorr",
        "description": "Focal gamma=1.0 + Decorrelation 0.01",
        "env": {
            "USE_AUX_LOSSES": "1",
            "USE_FOCAL_LOSS": "1", "FOCAL_GAMMA": "1.0",
            "LAMBDA_DECORR": "0.01",
            "LAMBDA_RANK": "0",
            "LAMBDA_UNIGRAM": "0"
        }
    },
    {
        "name": "stack_all",
        "description": "All winning losses combined",
        "env": {
            "USE_AUX_LOSSES": "1",
            "USE_FOCAL_LOSS": "1", "FOCAL_GAMMA": "1.0",
            "LAMBDA_DECORR": "0.01",
            "LAMBDA_RANK": "0.05", "RANK_EVERY": "10",
            "LAMBDA_UNIGRAM": "0.1"
        }
    },
]

# Write a temporary config with stacking experiments
cfg['experiments'] = stack_experiments
with open('experiments/sweep_config_stack.json', 'w') as f:
    json.dump(cfg, f, indent=2)

!python3 scripts/experiment_runner.py --config experiments/sweep_config_stack.json --seeds 42,1337,7

## 6. Final Report + Save to Drive

In [ ]:
%cd /content/parameter-golf
!python3 scripts/experiment_runner.py --report-only

import shutil
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'

# Save report and JSON to Drive
for f in ['experiments/results_auto.md', 'experiments/results_auto.json']:
    if os.path.exists(f):
        shutil.copy2(f, f'{DRIVE_DIR}/results/{os.path.basename(f)}')
        print(f'Saved {f} → Drive')

# Display
from IPython.display import Markdown, display
if os.path.exists('experiments/results_auto.md'):
    display(Markdown(open('experiments/results_auto.md').read()))

## 7. Custom Run (manual)

Run any experiment manually with full control over env vars.

In [ ]:
# Example: custom experiment with specific settings
# %cd /content/parameter-golf
#
# !RUN_ID=custom_test SEED=42 \
#  ITERATIONS=500 TRAIN_BATCH_TOKENS=786432 VAL_LOSS_EVERY=100 \
#  MAX_WALLCLOCK_SECONDS=300 \
#  USE_AUX_LOSSES=1 USE_FOCAL_LOSS=1 FOCAL_GAMMA=1.0 \
#  LAMBDA_DECORR=0.01 LAMBDA_RANK=0.05 RANK_EVERY=10 \
#  LAMBDA_UNIGRAM=0.1 \
#  python3 train_gpt_aux.py

## 7b. Scheduled Perturbation Experiments (1 hour each)

Tests convergence acceleration through the training plateau (steps 500-3500).
These run on the COMPILED path with zero overhead — same speed as baseline.

- `baseline_1h`: reference at 1-hour wallclock
- `sched_smooth_01`: label smoothing peak=0.1 during plateau
- `sched_noise_005`: gradient noise 0.005 during plateau
- `sched_combo`: both combined

In [ ]:
# Scheduled perturbation experiments — top 3 + baseline (~4 hours total)
%cd /content/parameter-golf
!python3 scripts/experiment_runner.py --config {CONFIG} \
    --filter "baseline_1h|sched_smooth_01|sched_noise_005|sched_combo" --seeds 42

In [ ]:
# Full perturbation sweep (all 7 variants, ~7 hours)
# Run this after the top-3 screening if any showed improvement
# %cd /content/parameter-golf
# !python3 scripts/experiment_runner.py --config {CONFIG} --filter "sched_" --seeds 42

In [ ]:
# Top-K margin experiments (uses forward_aux, slower — run only if perturbations didn't help)
# %cd /content/parameter-golf
# !python3 scripts/experiment_runner.py --config {CONFIG} --filter "margin|boost" --seeds 42

## 7c. CharacterHash Experiments (1 hour each)

Tests character-level morphological features — ORTHOGONAL information the model
currently has zero access to. Addresses the word-boundary bottleneck (46.5% of errors).

Fits in the existing 84KB artifact budget. No layers shrunk, no width reduced.

In [ ]:
# CharacterHash experiments — 3 variants + baseline_1h (~4 hours)
%cd /content/parameter-golf
!python3 scripts/experiment_runner.py --config {CONFIG} \
    --filter "baseline_1h|char_hash" --seeds 42

## 8. Error Analysis — Where Does the Model Fail?

Train a baseline model and analyze its per-token error patterns.
This tells us WHERE the model struggles so we can design targeted losses.

**Step 1:** Train a baseline (skip if you already have `final_model.pt` from Step 1)

In [ ]:
# Train baseline to full saturation for error analysis
# ~5 hours on RTX PRO 6000 without torch.compile (~7100 steps, matches SOTA)
%cd /content/parameter-golf

import os
if not os.path.exists('final_model.pt'):
    print("Training baseline model for error analysis (~5 hours, full SOTA equivalent)...")
    !USE_AUX_LOSSES=0 USE_COMPILE=0 RUN_ID=error_analysis_baseline SEED=42 \
     ITERATIONS=20000 TRAIN_BATCH_TOKENS=786432 VAL_LOSS_EVERY=500 \
     MAX_WALLCLOCK_SECONDS=18000 \
     python3 train_gpt_aux.py
else:
    print(f"Model checkpoint found: final_model.pt ({os.path.getsize('final_model.pt') / 1e6:.1f} MB)")
    import glob
    logs = sorted(glob.glob('logs/error_analysis_baseline*.txt'))
    if logs:
        with open(logs[-1]) as f:
            lines = f.readlines()
        for line in lines[-10:]:
            if 'step:' in line or 'val_bpb' in line or 'peak memory' in line:
                print(line.strip())

In [ ]:
# Run error analysis — takes ~5 min on A100
%cd /content/parameter-golf
!python3 scripts/error_analysis.py \
    --model final_model.pt \
    --max-sequences 500 \
    --batch-size 8 \
    --output experiments/error_analysis.json

# Save to Drive
import shutil
shutil.copy2('experiments/error_analysis.json', f'{DRIVE_DIR}/results/error_analysis.json')
print(f"\nSaved to Drive: {DRIVE_DIR}/results/error_analysis.json")